# Bronze Layer Ingestion Pipeline - Generic & Metadata-Driven

Generic notebook to ingest ANY source into Bronze Delta tables using LakeForge framework configuration.

**Supported input formats:** csv, tsv, psv, json, jsonl/ndjson, parquet, avro, orc, xml, xlsx/xls (Excel), delta, and plain/fixed-width text — selected in config or auto-detected from the file extension via the UniversalLoader.

In [0]:
%pip install openpyxl chardet


In [0]:
# Notebook Parameters
dbutils.widgets.text("config_dir", "", "Config Directory (Optional - Auto-resolved if blank)")
dbutils.widgets.text("source_system", "erp", "Source System")
dbutils.widgets.text("pipeline_name", "customers", "Pipeline (Source) Name")
dbutils.widgets.dropdown("load_type", "full", ["full", "incremental"], "Load Type")
dbutils.widgets.dropdown("environment", "dev", ["dev", "test", "staging", "prod"], "Environment / Stage")
dbutils.widgets.text("client_id", "", "Client ID / Tenant (Optional)")
dbutils.widgets.dropdown("create_views", "false", ["false", "true"], "Create Views (Optional)")

# Resolve config_dir and repo_root generically
import os
import sys
from pathlib import Path

raw_config_dir = ""
try:
    raw_config_dir = dbutils.widgets.get("config_dir").strip()
except Exception:
    raw_config_dir = ""

if raw_config_dir:
    config_dir = raw_config_dir
    repo_root = str(Path(config_dir).parent)
else:
    try:
        # Dynamically discover the notebook's workspace/repo location
        nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        repo_root = str(Path(nb_path).parent.parent)
        config_dir = str(Path(repo_root) / "configs")
    except Exception:
        cwd = Path(os.getcwd())
        if (cwd / "configs").exists():
            repo_root = str(cwd)
            config_dir = str(cwd / "configs")
        elif (cwd.parent / "configs").exists():
            repo_root = str(cwd.parent)
            config_dir = str(cwd.parent / "configs")
        else:
            repo_root = str(cwd)
            config_dir = "configs"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

source_system = dbutils.widgets.get("source_system")
pipeline_name = dbutils.widgets.get("pipeline_name")
load_type = dbutils.widgets.get("load_type")
try:
    _stage = dbutils.widgets.get("stage").strip()
except Exception:
    _stage = ""
environment = _stage or dbutils.widgets.get("environment")
client_id = dbutils.widgets.get("client_id").strip() or None
create_views = dbutils.widgets.get("create_views").lower() == "true"

print(f"Executing Bronze Ingestion for: {pipeline_name} in {environment} (client: {client_id or 'global'}, load_type: {load_type})")


In [0]:
# Imports & Path setup
import sys

from dataclasses import asdict
from lakeforge import (
    ConfigParser,
    create_dq_engine,
    create_bronze_writer,
    create_trust_engine
)

from pyspark.sql import functions as F
from datetime import datetime

print("✅ Framework libraries loaded")

In [0]:
# Load Configurations
ing_config = ConfigParser.get_ingestion_config(config_dir, pipeline_name, environment, client_id=client_id)
dq_config = ConfigParser.get_dq_config(config_dir, pipeline_name, environment, client_id=client_id)

target_table = f"{ing_config.target_catalog}.{ing_config.target_schema}.{ing_config.target_table}"
print(f"Target Table: {target_table}")


In [0]:
# Load Source Data dynamically - ANY format
# Supported: csv, tsv, psv, json, jsonl/ndjson, parquet, avro, orc, xml, xlsx/xls (Excel), delta, txt/fixed-width
# The format comes from config (file_format / source_type) or is auto-detected from the file extension.
from lakeforge.ingestion import create_universal_loader

universal_loader = create_universal_loader(spark)
df_raw = universal_loader.load_from_config(ing_config)

print(f"✅ Raw data loaded from: {ing_config.source_path}")
print(f"   Format: {ing_config.file_format or ing_config.source_type} | Rows: {df_raw.count()}")

In [0]:
# Run Data Quality Engine validations
dq_engine = create_dq_engine(spark)
rules_list = [asdict(r) for r in dq_config.rules]

dq_results = dq_engine.validate_dataframe(
    df=df_raw,
    rules=rules_list,
    quarantine_failures=True
)

print(f"DQ Execution Summary: {dq_results['rules_passed']}/{dq_results['rules_executed']} rules passed")

In [0]:
# Segregate and write clean / quarantine records
business_keys = ing_config.business_key

if dq_results["quarantine_count"] > 0:
    # Left anti join if business_keys provided, otherwise subtract
    if business_keys:
        clean_df = df_raw.join(dq_results["quarantine_df"], on=business_keys, how="left_anti")
    else:
        clean_df = df_raw.subtract(dq_results["quarantine_df"])
else:
    clean_df = df_raw

# Write clean records
bronze_writer = create_bronze_writer(spark)
bronze_writer.write_to_bronze(
    df=clean_df,
    target_table=ing_config.target_table,
    catalog=ing_config.target_catalog,
    schema=ing_config.target_schema,
    mode=ing_config.mode,
    merge_keys=business_keys,
    add_audit_columns=True,
    source_system=source_system
)
print(f"✅ Wrote {clean_df.count()} clean rows to {target_table}")

# Write quarantine records if any
if dq_results["quarantine_count"] > 0:
    dq_engine.write_quarantine_table(
        quarantine_df=dq_results["quarantine_df"],
        catalog=dq_config.catalog,
        schema=dq_config.schema,
        table_name=dq_config.quarantine_table,
        validation_results=dq_results
)
    print(f"⚠️ Quarantined {dq_results['quarantine_count']} rows in {dq_config.quarantine_table}")

In [0]:
# Compute and Log Trust Score
trust_engine = create_trust_engine(spark)
trust_score = trust_engine.calculate_trust_score(
    table_name=target_table,
    dq_results=dq_results,
    pipeline_stage="bronze"
)

print(f"🎯 Pipeline Trust Score: {trust_score['overall_score']:.1f}%")
print(f"🎯 Trust Level: {trust_score['trust_level']}")

In [0]:
# Create Views if explicitly enabled (otherwise run VIEWS_Creation_Pipeline on-demand)
if create_views:
    try:
        print("Checking for configured views to create...")
        from lakeforge.views.view_manager import create_view_manager
        from lakeforge.metadata.config_parser import load_view_configs
        env_config = ConfigParser.get_environment_config(config_dir, environment)
        view_configs = load_view_configs(config_dir, environment, client_id=client_id)
        target_views = [vc for vc in view_configs if vc.schema == env_config["bronze_schema"]]
        if target_views:
            view_manager = create_view_manager(spark, env_config)
            view_manager.create_views(target_views)
            print(f"Created {len(target_views)} views successfully")
        else:
            print("No views configured for bronze schema")
    except Exception as e:
        print(f"Error creating views: {str(e)}")
else:
    print("View creation skipped (decoupled; trigger VIEWS_Creation_Pipeline on demand or set create_views=true)")
